# Drone 2023 + 2024: Data Store → flight products → bulk results → Data Store

Run from this repository in your cloud container. Use a persistent disk. The notebook processes one ExportPackage at a time, retains durable outputs, and resumes using the pipeline's own checkpoints. It does not download NEON data.

**Use the repaired repository checkout**, including `scripts/drone_cloud_workflow.py` and the drone spatial fixes, in your container. These local changes must be transferred or committed/pushed before cloning them there. The missing spatial repair has now been recovered from the GitHub Desktop stash. Install with `python -m pip install -e .` in the notebook kernel environment. Install the Linux GoCommands client and run `gocmd init` in a terminal; credentials never belong in notebook cells.

This is an explicit research recovery recipe: use the original TIFF grid; retain ten declared TIFF bands only after exact comparison with the H5; preserve terrain masks and measured constant view angles. An unknown eleventh channel and acquisition-date discrepancies remain unresolved (P149). Review scientific QA before treating outputs as approved results. Unsupported packages stop rather than receive guessed metadata.

## 1. Paths and choices
Keep `RUN_ID` fixed when resuming. Use a new ID for a different source recipe or campaign. The upload directory is dedicated to this run, separate from original sources.


In [ ]:
from pathlib import Path
import sys

REPO = Path.cwd().resolve()  # Open this notebook from the spectralbridge repo root.
WORK_PARENT = Path("/home/jovyan/data-store/spectralbridge_drone")
RUN_ID = "drone_2023_2024_spatial_recovery_v1"
REMOTE_PARENT = "i:/iplant/home/shared/earthlab/macrosystems/field-data/output"
SOURCES = {year: f"{REMOTE_PARENT}/summer-{year}-10cm-10k" for year in (2023, 2024)}
REMOTE_RESULTS = f"{REMOTE_PARENT}/spectralbridge_processed/{RUN_ID}"
GOCMD = "gocmd"  # Or the absolute Linux executable path.
POLYGONS = REPO / "Datasets/niwot_aop_polygons_2023_12_8_23_analysis_ready_half_diam.gpkg"
MANIFEST = REPO / "src/spectralbridge/data/drone_field_manifest.csv"
REFLECTANCE_BANDS = tuple(range(1, 11))  # Explicit retained-band recipe, not automatic trimming.
TEST_PACKAGE = None  # Or "AOP-Ruby-08-14-23-ExportPackage" for an explicitly labeled one-flight test.
INSTALL_PACKAGE = False  # Set True once if this kernel has not installed this checkout.
CLEANUP_STAGED_INPUTS = False  # Only after that flight passes producer/readiness gates.
RUN = False  # Set True after reviewing paths; then run the remaining cells in order.
UPLOAD = False  # Set True to enable the final checksum-verified uploads.
WORK = WORK_PARENT / RUN_ID


## 2. Load this checkout and check the spatial repair
Set `INSTALL_PACKAGE=True` once if needed. The install command uses this kernel's Python. It does not pull or reset your checkout. Restart the kernel after installation if it previously imported SpectralBridge.


In [ ]:
import subprocess
assert (REPO / "scripts/drone_cloud_workflow.py").is_file(), "Copy the new notebook AND repaired repository to the container."
if RUN and INSTALL_PACKAGE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO)], check=True)


In [ ]:
import inspect
import json
import shutil
import spectralbridge.pipelines.drone as drone
from spectralbridge import inspect_drone_collection, run_bulk_pipeline, summarize_bulk_results
from spectralbridge.drone_production import validate_bulk_ready_flightline
from spectralbridge.bulk.results import BulkResultsConfig
sys.path.insert(0, str(REPO / "scripts"))
from drone_cloud_workflow import (
    JsonGocmdBackend, download_sources, prepare_reference,
    require_complete, select_inventory, upload_tree, write_json, CAUTIONS,
)

assert RUN, "Review configuration, set RUN=True, then rerun from the configuration cell."
assert Path(drone.__file__).resolve().is_relative_to(REPO / "src"), "Kernel is using another package checkout. Restart after editable install."
assert hasattr(drone, "_drone_envi_header"), "Drone spatial repair is missing from this checkout."
assert hasattr(drone, "_drone_grid_transforms_match"), "Ancillary roundoff validation is missing from this checkout."
assert "envi_corner_v1" in inspect.getsource(drone.convert_drone_tiff_to_h5), "TIFF corner-convention repair is missing."
assert POLYGONS.is_file() and MANIFEST.is_file()
WORK.mkdir(parents=True, exist_ok=True)
STATE = WORK / "campaign_state"
FLIGHTS = WORK / "flight_products"
STATE.mkdir(exist_ok=True)
FLIGHTS.mkdir(exist_ok=True)
backend = JsonGocmdBackend(GOCMD)
backend.ensure_available()
print("Loaded:", drone.__file__)
print("Free disk GiB:", round(shutil.disk_usage(WORK).free / 2**30, 1))
print("Source/provenance cautions:", *CAUTIONS, sep="\n- ")


## 3. Discover both years before downloading
Review year summaries and exclusions. Completeness follows the acquisition manifest, not a hard-coded number of flights. The JSON adapter handles the current GoCommands listing format. A missing year or expected flight prevents production bulk analysis. `TEST_PACKAGE` explicitly limits an integration test to one manifest-matched package; its saved configuration labels the subset, and its results cannot establish campaign-wide transferability. Set it back to `None` for production and use a different run ID.


In [ ]:
inventory = inspect_drone_collection(
    SOURCES, years=sorted(SOURCES), manifest=MANIFEST,
    backend=backend, inventory_output_dir=STATE / "inventory",
)
full_inventory = inventory
inventory = select_inventory(full_inventory, TEST_PACKAGE)
inventory.write(STATE / "selected_inventory")
print("Scope:", "ONE-FLIGHT INTEGRATION TEST" if TEST_PACKAGE else "COMPLETE CAMPAIGN")
print(json.dumps(inventory.year_summaries(), indent=2, default=str))
print("Excluded:", [(p.package_name, p.reason) for p in inventory.excluded_packages])
assert inventory.discovery_complete and inventory.eligible_packages, "Resolve missing sources/manifest flights before running."
write_json(STATE / "run_configuration.json", {
    "sources": SOURCES, "run_id": RUN_ID, "bands": REFLECTANCE_BANDS,
    "scope": "one_flight_integration_test" if TEST_PACKAGE else "complete_campaign", "test_package": TEST_PACKAGE,
    "manifest": str(MANIFEST), "polygons": str(POLYGONS), "cautions": CAUTIONS,
    "code_revision": subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip(),
    "working_tree_status": subprocess.check_output(["git", "status", "--short"], cwd=REPO, text=True),
})


## 4. Download → verify → process each flight
Uses `aligned_orthomosaic.tif`, `aligned_slope.tif`, `aligned_aspect.tif`, and the inventoried original H5 in each ExportPackage. The TIFF supplies geographic placement; the H5 supplies verified retained-band values, spectral metadata and constant view angles. Original files are never edited.

Each package gets a separate output folder. Source hashes and the declared band recipe are saved with its durable products. Processing failures are recorded while remaining flights continue. Reexecute this cell to retry; the pipeline revalidates its checkpoints. Staged sources are retained by default to preserve checkpoint fingerprints on restart. Enabling cleanup saves disk after success but requires downloading and preparing those inputs again on a rerun. Working H5s and all processed products remain on disk. Budget persistent storage for the entire output campaign and one staged package.


In [ ]:
statuses = {}  # Rebuild the population from this run, never a stale success list.
for number, package in enumerate(inventory.eligible_packages, 1):
    key = package.remote_package_path
    label = f"{package.year}_{package.package_name}"
    staging = WORK / "staged_sources" / label
    prepared = WORK / "prepared_sources" / str(package.year) / package.package_name
    output = FLIGHTS / label
    print(f"[{number}/{len(inventory.eligible_packages)}] {label}", flush=True)
    try:
        download_sources(package, staging, backend)
        reference, tiff_options = prepare_reference(
            staging, package.h5_name, prepared, REFLECTANCE_BANDS,
        )
        output.mkdir(parents=True, exist_ok=True)
        shutil.copy2(prepared / "source_recipe.json", output / "source_recipe.json")
        result = drone.run_drone_pipeline(
            reference, polygon_path=POLYGONS, output_dir=output,
            drone_manifest_path=MANIFEST, drone_manifest_timezone="UTC",
            apply_topo=True, apply_brdf=True, require_solar_geometry=True,
            apply_translation=True, extraction_mode="polygon",
            merge_extractions=False, overwrite=False, raise_on_incomplete=True,
            **tiff_options,
        )
        write_json(output / "pipeline_result.json", result)
        if result.get("failed") or result.get("blocked"):
            raise RuntimeError("Pipeline reported failed or blocked products")
        flight_dirs = [Path(record["flight_dir"]) for record in result["qa_summary"]["files"]]
        if len(flight_dirs) != 1:
            raise RuntimeError("Expected exactly one produced flight per ExportPackage")
        readiness = validate_bulk_ready_flightline(flight_dirs[0])
        if not readiness.bulk_ready:
            raise RuntimeError(f"Producer-to-bulk validation failed: {readiness}")
        statuses[key] = {"bulk_ready": True, "flight_dir": str(flight_dirs[0]), "package": label}
        if CLEANUP_STAGED_INPUTS:
            shutil.rmtree(prepared)
            shutil.rmtree(staging)
    except Exception as exc:
        statuses[key] = {"bulk_ready": False, "package": label, "error": str(exc)}
        print("FAILED:", label, exc, flush=True)
    write_json(STATE / "status.json", statuses)
print(json.dumps(statuses, indent=2))


## 5. Gate the whole campaign, then run bulk and summaries
Bulk reads only validated completed flights. The separate bulk-input directory contains links to exactly the current population; it is temporary and is not uploaded. No pixel-scale bulk cache is materialized. Failed or missing individuals must be resolved before proceeding.


In [ ]:
require_complete(inventory, statuses)
paths = [Path(statuses[p.remote_package_path]["flight_dir"]) for p in inventory.eligible_packages]
for path in paths:
    assert validate_bulk_ready_flightline(path).bulk_ready, f"Flight changed: {path}"
assert len({p.resolve() for p in paths}) == len(paths), "Duplicate flight paths"
BULK_INPUT = WORK / "bulk_inputs"
BULK_INPUT.mkdir(exist_ok=True)
for child in BULK_INPUT.iterdir():
    if not child.is_symlink():
        raise RuntimeError(f"Unexpected real file in temporary bulk-input directory: {child}")
    child.unlink()
for index, path in enumerate(paths):
    (BULK_INPUT / f"{index:04d}_{path.name}").symlink_to(path.resolve(), target_is_directory=True)
bulk_options = dict(
    input_mode="flightline_outputs", analysis="translation", on_invalid="exclude",
    materialize_observations=False, extraction_workers=1,
    spectral_library=None, make_summary_plots=False, make_full_spectral_reports=False,
)
preflight = run_bulk_pipeline(BULK_INPUT, WORK / "bulk_preflight", preflight_only=True, **bulk_options)
write_json(STATE / "bulk_preflight_result.json", preflight)
assert preflight["accepted_flightline_count"] == len(paths), "Bulk excluded a flight; inspect preflight before continuing."
bulk_result = run_bulk_pipeline(BULK_INPUT, WORK / "bulk_analysis", **bulk_options)
write_json(STATE / "bulk_result.json", bulk_result)
summary = summarize_bulk_results(
    WORK / "bulk_analysis", config=BulkResultsConfig(), make_figures=True, make_report=True,
)
write_json(STATE / "summary_result.json", summary)
print("Bulk summary status:", summary["status"])
print("Report:", summary["report_pdf"])


## 6. Upload flight products, compact bulk outputs and provenance
Set `UPLOAD=True` in configuration and rerun configuration plus this cell after inspecting QA. Uploads use a dedicated run collection, hash-based difference checking, and transfer checksum verification. Rerunning resumes transfers to that same run destination. A changed local file replaces its counterpart within this run; keep the run ID fixed only while resuming the same campaign.

Uploads include native/corrected/translated products, extraction tables, QA, working metadata, compact bulk checkpoints/results, figures/reports and campaign receipts. Original downloaded sources, prepared TIFFs and temporary bulk-input links are excluded. Each uploaded directory receives a local SHA-256 file manifest. This does not certify scientific accuracy; preserve the recorded cautions with results.


In [ ]:
require_complete(inventory, statuses)
assert "summary" in globals() and "bulk_result" in globals(), "Complete the bulk/summaries cell first."
if UPLOAD:
    destinations = []
    for folder in [FLIGHTS, WORK / "bulk_preflight", WORK / "bulk_analysis", STATE]:
        assert folder.is_dir(), folder
        destinations.append(upload_tree(folder, REMOTE_RESULTS, GOCMD))
    write_json(WORK / "upload_receipt.json", {"destinations": destinations, "verification": "gocmd --verify_checksum; local SHA-256 manifests", "cautions": CAUTIONS})
    print("Uploaded:", *destinations, sep="\n")
else:
    print("Uploads disabled. Review outputs at", WORK, "then enable UPLOAD.")
